# FULL_DOC_MARKDOWN 文件关系图召回实验

目标：对 `nyonic-openviking/FULL_DOC_MARKDOWN` 下的 Markdown 文件做文件级 embedding，构建一个无向关系图，并在 query 时用“query 向量相似度 + 图上的 A* 风格扩展”返回 top-k 关联文件。

设计要点：

- 每个文件对应一个图节点，节点向量来自文件路径 + 文本内容。
- 文件之间按 cosine similarity 建边；全目录统计边相似度后做 min-max 归一化。
- A* 中边 cost = `1 - normalized_similarity + epsilon`，越相关的文件越容易被走到。
- 查询时先用 query embedding 找入口 seed，再在图中扩展，综合直接相似度和路径代价排序。
- notebook 默认优先使用本地 `Qwen3-Embedding-0.6B`；embedding 和图关系都会写缓存，配置和文件未变化时不会重复计算。

## 依赖

如果缺依赖，先在当前环境安装：

```bash
pip install numpy pandas scikit-learn tqdm openai sentence-transformers
```

Embedding 后端可用环境变量切换：

- `EMBEDDING_BACKEND`：`openai` / `local` / `tfidf`
- `EMBEDDING_API_KEY` 或 `OPENAI_API_KEY`
- `EMBEDDING_API_BASE`，可选，例如本地或第三方兼容服务
- `EMBEDDING_MODEL`，OpenAI-compatible 默认 `text-embedding-3-small`
- `LOCAL_EMBEDDING_MODEL`，本地默认优先读取 `models/Qwen3-Embedding-0.6B`，不存在时使用 `Qwen/Qwen3-Embedding-0.6B`

## 清理缓存

默认不会清理任何缓存。需要重建时，把下面对应开关改成 `True` 后运行这一格。这个版本只删除固定缓存文件，不递归扫描目录。

In [1]:
from pathlib import Path

CACHE_DIR_FOR_CLEANUP = Path('graph_recall_cache')
CACHE_FILE_NAMES = [
    'embedding_fingerprint.json',
    'document_embeddings.npy',
    'documents.jsonl',
    'tfidf_vectorizer.joblib',
    'document_vectors.faiss',
    'vector_store_fingerprint.json',
    'graph_fingerprint.json',
    'relation_edges.jsonl',
]

# 默认只显示将要清理的文件名，不触碰文件系统。
# 确认要删除时，把 DRY_RUN 改成 False，并把对应 CLEAR_* 改成 True。
DRY_RUN = False
CLEAR_EMBEDDING_CACHE = True
CLEAR_GRAPH_CACHE = True

EMBEDDING_CACHE_FILES = [
    'embedding_fingerprint.json',
    'document_embeddings.npy',
    'documents.jsonl',
    'tfidf_vectorizer.joblib',
    'document_vectors.faiss',
    'vector_store_fingerprint.json',
]
GRAPH_CACHE_FILES = [
    'graph_fingerprint.json',
    'relation_edges.jsonl',
]

names = []
if CLEAR_EMBEDDING_CACHE:
    names.extend(EMBEDDING_CACHE_FILES)
if CLEAR_GRAPH_CACHE:
    names.extend(GRAPH_CACHE_FILES)

removed = []
if names and not DRY_RUN:
    for name in names:
        path = CACHE_DIR_FOR_CLEANUP / name
        try:
            path.unlink(missing_ok=True)
            removed.append(name)
        except PermissionError:
            print('permission/file-lock skip:', path)

print('cache dir:', CACHE_DIR_FOR_CLEANUP)
print('known cache files:', CACHE_FILE_NAMES)
print('selected:', names if names else 'none')
print('dry run:', DRY_RUN)
print('removed:', removed if removed else 'none')


cache dir: graph_recall_cache
known cache files: ['embedding_fingerprint.json', 'document_embeddings.npy', 'documents.jsonl', 'tfidf_vectorizer.joblib', 'document_vectors.faiss', 'vector_store_fingerprint.json', 'graph_fingerprint.json', 'relation_edges.jsonl']
selected: ['embedding_fingerprint.json', 'document_embeddings.npy', 'documents.jsonl', 'tfidf_vectorizer.joblib', 'document_vectors.faiss', 'vector_store_fingerprint.json', 'graph_fingerprint.json', 'relation_edges.jsonl']
dry run: False
removed: ['embedding_fingerprint.json', 'document_embeddings.npy', 'documents.jsonl', 'tfidf_vectorizer.joblib', 'document_vectors.faiss', 'vector_store_fingerprint.json', 'graph_fingerprint.json', 'relation_edges.jsonl']


In [2]:
from __future__ import annotations

import hashlib
import heapq
import importlib.util
import json
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT_DIR = Path('FULL_DOC_MARKDOWN')
CACHE_DIR = Path('graph_recall_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_EXTENSIONS = {'.md', '.markdown'}
LOW_VRAM_MODE = os.getenv('LOW_VRAM_MODE', '1').lower() not in {'0', 'false', 'no'}
MAX_FILE_BYTES = int(os.getenv('MAX_FILE_BYTES', str(256 * 1024 if LOW_VRAM_MODE else 1024 * 1024)))
# 低显存模式下只取较短文本，显著降低 activation 峰值。
MAX_EMBED_CHARS = int(os.getenv('MAX_EMBED_CHARS', '4000' if LOW_VRAM_MODE else '8000'))
TEXT_HEAD_CHARS = int(os.getenv('TEXT_HEAD_CHARS', '1700' if LOW_VRAM_MODE else str(MAX_EMBED_CHARS)))
TEXT_TAIL_CHARS = int(os.getenv('TEXT_TAIL_CHARS', '500' if LOW_VRAM_MODE else '0'))

def package_available(package_name: str) -> bool:
    return importlib.util.find_spec(package_name) is not None


# local: 本地 sentence-transformers；openai: 调 embedding API；tfidf: 保底验证流程。
REQUESTED_EMBEDDING_BACKEND = os.getenv('EMBEDDING_BACKEND')
HAS_EMBEDDING_API_KEY = bool(os.getenv('EMBEDDING_API_KEY') or os.getenv('OPENAI_API_KEY'))
HAS_SENTENCE_TRANSFORMERS = package_available('sentence_transformers')
LOCAL_MODEL_PATH = Path('models/Qwen3-Embedding-0.6B')
DEFAULT_LOCAL_EMBEDDING_MODEL = str(LOCAL_MODEL_PATH) if LOCAL_MODEL_PATH.exists() else 'Qwen/Qwen3-Embedding-0.6B'
EMBEDDING_BACKEND = REQUESTED_EMBEDDING_BACKEND or (
    'local' if HAS_SENTENCE_TRANSFORMERS else ('openai' if HAS_EMBEDDING_API_KEY else 'tfidf')
)
OPENAI_EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')
LOCAL_EMBEDDING_MODEL = os.getenv('LOCAL_EMBEDDING_MODEL', DEFAULT_LOCAL_EMBEDDING_MODEL)
EMBEDDING_MODEL = OPENAI_EMBEDDING_MODEL if EMBEDDING_BACKEND == 'openai' else (
    LOCAL_EMBEDDING_MODEL if EMBEDDING_BACKEND == 'local' else 'tfidf'
)
DEFAULT_BATCH_SIZE = '1' if EMBEDDING_BACKEND == 'local' else '64'
EMBEDDING_BATCH_SIZE = int(os.getenv('EMBEDDING_BATCH_SIZE', DEFAULT_BATCH_SIZE))
LOCAL_EMBEDDING_MAX_SEQ_LENGTH = int(os.getenv('LOCAL_EMBEDDING_MAX_SEQ_LENGTH', '512' if LOW_VRAM_MODE else '2048'))
LOCAL_EMBEDDING_TORCH_DTYPE = os.getenv('LOCAL_EMBEDDING_TORCH_DTYPE', 'float16' if LOW_VRAM_MODE else '')

# 默认全量构建；试跑时可设置环境变量 MAX_FILES=200。
MAX_FILES_ENV = os.getenv('MAX_FILES')
if MAX_FILES_ENV and MAX_FILES_ENV.lower() in {'none', 'all', 'full'}:
    MAX_FILES: int | None = None
elif MAX_FILES_ENV:
    MAX_FILES = int(MAX_FILES_ENV)
else:
    MAX_FILES = 500 if EMBEDDING_BACKEND == 'tfidf' else None

NEIGHBOR_K = 8
MIN_EDGE_SIMILARITY = 0.35
SIMILARITY_BLOCK_SIZE = 512

print('root:', ROOT_DIR.resolve())
print('cache:', CACHE_DIR.resolve())
print('sentence-transformers available:', HAS_SENTENCE_TRANSFORMERS)
print('local model path exists:', LOCAL_MODEL_PATH.exists(), '|', LOCAL_MODEL_PATH.resolve())
print('low vram mode:', LOW_VRAM_MODE)
print('embedding batch size:', EMBEDDING_BATCH_SIZE)
print('local max seq length:', LOCAL_EMBEDDING_MAX_SEQ_LENGTH)
print('local torch dtype:', LOCAL_EMBEDDING_TORCH_DTYPE or 'model default')
print('max embed chars:', MAX_EMBED_CHARS)
print('max files:', MAX_FILES if MAX_FILES is not None else 'all')
print('embedding backend:', EMBEDDING_BACKEND, '| model:', EMBEDDING_MODEL)

root: D:\nyonic\minidemo\Dir_graph_recall\FULL_DOC_MARKDOWN
cache: D:\nyonic\minidemo\Dir_graph_recall\graph_recall_cache
sentence-transformers available: True
local model path exists: True | D:\nyonic\minidemo\Dir_graph_recall\models\Qwen3-Embedding-0.6B
low vram mode: True
embedding batch size: 1
local max seq length: 512
local torch dtype: float16
max embed chars: 4000
max files: all
embedding backend: local | model: models\Qwen3-Embedding-0.6B


c:\Users\forsa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 扫描与文本构造

这里采用“文件级节点”：每个 Markdown 文件生成一个向量。为避免超长文件拖慢 embedding，请用 `MAX_EMBED_CHARS` 截断输入；后续如果要做更细粒度召回，可以把同一套图逻辑扩展到 section/chunk 级节点。

In [3]:
def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode('utf-8', errors='ignore')).hexdigest()


def read_text_lossy(path: Path) -> str:
    data = path.read_bytes()
    if len(data) > MAX_FILE_BYTES:
        data = data[:MAX_FILE_BYTES]
    return data.decode('utf-8-sig', errors='ignore')


def build_embedding_text(root: Path, path: Path, text: str) -> str:
    rel = path.relative_to(root).as_posix()
    title = path.stem.replace('_', ' ').replace('-', ' ')
    compact = '\n'.join(line.rstrip() for line in text.splitlines() if line.strip())
    if TEXT_TAIL_CHARS > 0 and len(compact) > MAX_EMBED_CHARS:
        head = compact[:TEXT_HEAD_CHARS]
        tail = compact[-TEXT_TAIL_CHARS:]
        compact = f'{head}\n\n...\n\n{tail}'
    compact = compact[:MAX_EMBED_CHARS]
    return f'Path: {rel}\nTitle: {title}\n\n{compact}'


def scan_documents(root: Path, max_files: int | None = None) -> list[dict[str, Any]]:
    if not root.exists():
        raise FileNotFoundError(f'ROOT_DIR not found: {root}')

    paths = sorted(
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in INCLUDE_EXTENSIONS
    )
    if max_files is not None:
        paths = paths[:max_files]

    docs: list[dict[str, Any]] = []
    for idx, path in enumerate(tqdm(paths, desc='reading markdown')):
        text = read_text_lossy(path)
        rel_path = path.relative_to(root).as_posix()
        embed_text = build_embedding_text(root, path, text)
        docs.append({
            'id': idx,
            'path': rel_path,
            'name': path.name,
            'size_bytes': path.stat().st_size,
            'content_sha256': sha256_text(text),
            'embedding_text': embed_text,
            'preview': text[:500].replace('\n', ' '),
        })
    return docs


docs = scan_documents(ROOT_DIR, MAX_FILES)
print(f'documents: {len(docs)}')
pd.DataFrame([{k: v for k, v in d.items() if k != 'embedding_text'} for d in docs]).head()

reading markdown: 100%|██████████| 9048/9048 [00:01<00:00, 7858.56it/s]

documents: 9048


,id,path,name,size_bytes,content_sha256,preview
0,0,CANoeCANalyzer/A429/A429.md,A429.md,1322,68ede0e69713fcf2dc6e575417cd4cbea2efa98b3b204b...,![](../../../Resources/vImages/vToolIcons/CANo...
1,1,CANoeCANalyzer/A429/A429Concept.md,A429Concept.md,735,170d8955b9d5445fea11ba9485f20ec4cc2c2d70721fc2...,A429 » Concept # Concept | Ove...
2,2,CANoeCANalyzer/A429/A429Features.md,A429Features.md,622,b888a0308458902c7dfba68a57b23b30f8d0db0a8f9d84...,A429 » Function Overview # Function Overv...
3,3,CANoeCANalyzer/A429/A429References.md,A429References.md,610,99e23169b6e5cbba0560dbcfa30de125c4292fd70f05b6...,A429 » References # References ...
4,4,CANoeCANalyzer/A429/basicsA429/A429basics.md,A429basics.md,3804,b9ae7a1c6806096c096bb36db08df44e07fde90a4087ff...,A429 » Basics # Basics ## ARIN...


## Embedding 后端

`embed_texts` 会返回 L2-normalized 矩阵，因此后续 `A @ B.T` 就是 cosine similarity。

In [4]:
def l2_normalize(matrix: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, eps)


_LOCAL_EMBEDDING_MODEL_INSTANCE = None


def get_local_embedding_model():
    global _LOCAL_EMBEDDING_MODEL_INSTANCE
    if _LOCAL_EMBEDDING_MODEL_INSTANCE is None:
        from sentence_transformers import SentenceTransformer

        kwargs = {}
        device = os.getenv('LOCAL_EMBEDDING_DEVICE')
        if device:
            kwargs['device'] = device
        if LOCAL_EMBEDDING_TORCH_DTYPE:
            try:
                import torch

                dtype_map = {
                    'float16': torch.float16,
                    'bfloat16': torch.bfloat16,
                    'float32': torch.float32,
                }
                kwargs['model_kwargs'] = {'torch_dtype': dtype_map[LOCAL_EMBEDDING_TORCH_DTYPE]}
            except Exception:
                kwargs.pop('model_kwargs', None)
        try:
            _LOCAL_EMBEDDING_MODEL_INSTANCE = SentenceTransformer(LOCAL_EMBEDDING_MODEL, **kwargs)
        except TypeError:
            kwargs.pop('model_kwargs', None)
            _LOCAL_EMBEDDING_MODEL_INSTANCE = SentenceTransformer(LOCAL_EMBEDDING_MODEL, **kwargs)
        if hasattr(_LOCAL_EMBEDDING_MODEL_INSTANCE, 'max_seq_length'):
            _LOCAL_EMBEDDING_MODEL_INSTANCE.max_seq_length = LOCAL_EMBEDDING_MAX_SEQ_LENGTH
    return _LOCAL_EMBEDDING_MODEL_INSTANCE


def release_cuda_cache() -> None:
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def embed_openai_compatible(texts: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> np.ndarray:
    from openai import OpenAI

    api_key = os.getenv('EMBEDDING_API_KEY') or os.getenv('OPENAI_API_KEY')
    base_url = os.getenv('EMBEDDING_API_BASE')
    if not api_key:
        raise RuntimeError('Missing EMBEDDING_API_KEY or OPENAI_API_KEY')

    client = OpenAI(api_key=api_key, base_url=base_url) if base_url else OpenAI(api_key=api_key)
    vectors: list[list[float]] = []
    for start in tqdm(range(0, len(texts), batch_size), desc='embedding api batches'):
        batch = texts[start:start + batch_size]
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        vectors.extend(item.embedding for item in resp.data)
    return l2_normalize(np.array(vectors, dtype=np.float32))


def embed_local_sentence_transformers(texts: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> np.ndarray:
    model = get_local_embedding_model()
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    release_cuda_cache()
    return l2_normalize(np.asarray(vectors, dtype=np.float32))


def embed_tfidf(texts: list[str]) -> tuple[Any, Any]:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize

    vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        max_features=12000,
    )
    matrix = vectorizer.fit_transform(texts)
    matrix = normalize(matrix, norm='l2', copy=False)
    return matrix, vectorizer


def cache_fingerprint(docs: list[dict[str, Any]]) -> dict[str, Any]:
    return {
        'backend': EMBEDDING_BACKEND,
        'model': EMBEDDING_MODEL,
        'low_vram_mode': LOW_VRAM_MODE,
        'max_file_bytes': MAX_FILE_BYTES,
        'max_embed_chars': MAX_EMBED_CHARS,
        'text_head_chars': TEXT_HEAD_CHARS,
        'text_tail_chars': TEXT_TAIL_CHARS,
        'local_embedding_max_seq_length': LOCAL_EMBEDDING_MAX_SEQ_LENGTH if EMBEDDING_BACKEND == 'local' else None,
        'local_embedding_torch_dtype': LOCAL_EMBEDDING_TORCH_DTYPE if EMBEDDING_BACKEND == 'local' else None,
        'root_dir': ROOT_DIR.as_posix(),
        'max_files': MAX_FILES,
        'docs': [
            {
                'path': d['path'],
                'content_sha256': d['content_sha256'],
                'embedding_text_sha256': sha256_text(d['embedding_text']),
            }
            for d in docs
        ],
    }


def same_fingerprint(a: dict[str, Any], b: dict[str, Any]) -> bool:
    return json.dumps(a, sort_keys=True, ensure_ascii=False) == json.dumps(b, sort_keys=True, ensure_ascii=False)


def build_or_load_embeddings(docs: list[dict[str, Any]], refresh: bool = False):
    fp_path = CACHE_DIR / 'embedding_fingerprint.json'
    emb_path = CACHE_DIR / 'document_embeddings.npy'
    meta_path = CACHE_DIR / 'documents.jsonl'
    tfidf_path = CACHE_DIR / 'tfidf_vectorizer.joblib'
    fp = cache_fingerprint(docs)

    if not refresh and fp_path.exists() and emb_path.exists():
        cached_fp = json.loads(fp_path.read_text(encoding='utf-8'))
        if same_fingerprint(fp, cached_fp):
            print('loading cached embeddings')
            embeddings = np.load(emb_path)
            vectorizer = None
            if EMBEDDING_BACKEND == 'tfidf' and tfidf_path.exists():
                import joblib
                vectorizer = joblib.load(tfidf_path)
            return embeddings, vectorizer

    texts = [d['embedding_text'] for d in docs]
    vectorizer = None
    if EMBEDDING_BACKEND == 'openai':
        embeddings = embed_openai_compatible(texts)
    elif EMBEDDING_BACKEND == 'local':
        embeddings = embed_local_sentence_transformers(texts)
    elif EMBEDDING_BACKEND == 'tfidf':
        embeddings, vectorizer = embed_tfidf(texts)
        embeddings = embeddings.astype(np.float32).toarray()
        import joblib
        joblib.dump(vectorizer, tfidf_path)
    else:
        raise ValueError(f'Unknown EMBEDDING_BACKEND: {EMBEDDING_BACKEND}')

    np.save(emb_path, embeddings.astype(np.float32))
    fp_path.write_text(json.dumps(fp, ensure_ascii=False, indent=2), encoding='utf-8')
    with meta_path.open('w', encoding='utf-8') as f:
        for doc in docs:
            slim = {k: v for k, v in doc.items() if k != 'embedding_text'}
            f.write(json.dumps(slim, ensure_ascii=False) + '\n')
    return embeddings, vectorizer


RUN_EMBEDDING_SMOKE_TEST = os.getenv('RUN_EMBEDDING_SMOKE_TEST', '0').lower() in {'1', 'true', 'yes'}
if RUN_EMBEDDING_SMOKE_TEST and EMBEDDING_BACKEND in {'local', 'openai'}:
    smoke_sentences = [
        'The weather is lovely today.',
        "It's so sunny outside!",
        'He drove to the stadium.',
    ]
    smoke_embeddings = (
        embed_local_sentence_transformers(smoke_sentences, batch_size=min(EMBEDDING_BATCH_SIZE, 3))
        if EMBEDDING_BACKEND == 'local'
        else embed_openai_compatible(smoke_sentences, batch_size=3)
    )
    smoke_similarities = smoke_embeddings @ smoke_embeddings.T
    print('smoke embeddings shape:', smoke_embeddings.shape)
    print('smoke similarities:')
    print(np.round(smoke_similarities, 4))


embeddings, tfidf_vectorizer = build_or_load_embeddings(docs)
print('embedding shape:', embeddings.shape)

`torch_dtype` is deprecated! Use `dtype` instead!
Batches: 100%|██████████| 9048/9048 [08:39<00:00, 17.40it/s]


embedding shape: (9048, 1024)


## FAISS 向量库缓存

这里把归一化后的文件向量写入 FAISS `IndexFlatIP`，用于直接向量召回和后续 agent skill 持久化加载。索引文件：`nyonic-openviking/graph_recall_cache/document_vectors.faiss`。

In [5]:
def build_or_load_faiss_index(vectors: np.ndarray, refresh: bool = False):
    import faiss

    index_path = CACHE_DIR / 'document_vectors.faiss'
    fp_path = CACHE_DIR / 'vector_store_fingerprint.json'
    embedding_fp_path = CACHE_DIR / 'embedding_fingerprint.json'
    embedding_fp = json.loads(embedding_fp_path.read_text(encoding='utf-8')) if embedding_fp_path.exists() else {}
    fp = {
        'kind': 'faiss.IndexFlatIP',
        'embedding_fingerprint': embedding_fp,
        'shape': list(vectors.shape),
        'dtype': str(vectors.dtype),
    }

    if not refresh and index_path.exists() and fp_path.exists():
        cached_fp = json.loads(fp_path.read_text(encoding='utf-8'))
        if same_fingerprint(fp, cached_fp):
            print('loading cached FAISS vector store')
            return faiss.read_index(str(index_path))

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors.astype(np.float32)))
    faiss.write_index(index, str(index_path))
    fp_path.write_text(json.dumps(fp, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved FAISS vector store:', index_path, '| vectors:', index.ntotal)
    return index


faiss_index = build_or_load_faiss_index(embeddings)
print('faiss ntotal:', faiss_index.ntotal)


def faiss_vector_search(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    qvec = embed_query(query).astype(np.float32).reshape(1, -1)
    scores, ids = faiss_index.search(np.ascontiguousarray(qvec), top_k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if 0 <= int(idx) < len(docs):
            results.append({
                'path': docs[int(idx)]['path'],
                'similarity': float(score),
                'preview': docs[int(idx)]['preview'],
            })
    return results

saved FAISS vector store: graph_recall_cache\document_vectors.faiss | vectors: 9048
faiss ntotal: 9048


## 构建无向关系图

每个节点只保留 top `NEIGHBOR_K` 个相似邻居，避免 9048 个文件形成过密的完全图。建边后对所有边的 similarity 做全局 min-max 归一化，再转为 A* 使用的 cost。

In [6]:
@dataclass(frozen=True)
class GraphEdge:
    source: int
    target: int
    similarity: float
    normalized_similarity: float
    cost: float


def top_similarity_edges(
    vectors: np.ndarray,
    neighbor_k: int = NEIGHBOR_K,
    min_similarity: float = MIN_EDGE_SIMILARITY,
    block_size: int = SIMILARITY_BLOCK_SIZE,
) -> list[tuple[int, int, float]]:
    n = vectors.shape[0]
    edge_best: dict[tuple[int, int], float] = {}

    for start in tqdm(range(0, n, block_size), desc='similarity blocks'):
        end = min(start + block_size, n)
        sims = vectors[start:end] @ vectors.T
        for local_i, row in enumerate(sims):
            i = start + local_i
            row[i] = -1.0
            k = min(neighbor_k, n - 1)
            if k <= 0:
                continue
            candidate_idx = np.argpartition(-row, kth=k - 1)[:k]
            for j in candidate_idx:
                sim = float(row[j])
                if sim < min_similarity:
                    continue
                a, b = sorted((i, int(j)))
                key = (a, b)
                if sim > edge_best.get(key, -1.0):
                    edge_best[key] = sim
    return [(a, b, sim) for (a, b), sim in edge_best.items()]


def normalize_edges(raw_edges: list[tuple[int, int, float]]) -> list[GraphEdge]:
    if not raw_edges:
        return []
    sims = np.array([edge[2] for edge in raw_edges], dtype=np.float32)
    sim_min = float(sims.min())
    sim_max = float(sims.max())
    denom = max(sim_max - sim_min, 1e-9)

    edges: list[GraphEdge] = []
    for source, target, sim in raw_edges:
        norm_sim = (sim - sim_min) / denom
        cost = 1.0 - norm_sim + 1e-4
        edges.append(GraphEdge(source, target, sim, float(norm_sim), float(cost)))
    return edges


def build_adjacency(edges: list[GraphEdge]) -> dict[int, list[tuple[int, float, float]]]:
    adjacency: dict[int, list[tuple[int, float, float]]] = {idx: [] for idx in range(len(docs))}
    for edge in edges:
        adjacency[edge.source].append((edge.target, edge.cost, edge.similarity))
        adjacency[edge.target].append((edge.source, edge.cost, edge.similarity))
    return adjacency


def graph_fingerprint() -> dict[str, Any]:
    fp_path = CACHE_DIR / 'embedding_fingerprint.json'
    embedding_fp = json.loads(fp_path.read_text(encoding='utf-8')) if fp_path.exists() else {}
    return {
        'embedding_fingerprint': embedding_fp,
        'neighbor_k': NEIGHBOR_K,
        'min_edge_similarity': MIN_EDGE_SIMILARITY,
        'similarity_block_size': SIMILARITY_BLOCK_SIZE,
    }


def build_or_load_graph(vectors: np.ndarray, refresh: bool = False):
    edges_path = CACHE_DIR / 'relation_edges.jsonl'
    fp_path = CACHE_DIR / 'graph_fingerprint.json'
    fp = graph_fingerprint()
    if not refresh and edges_path.exists() and fp_path.exists():
        cached_fp = json.loads(fp_path.read_text(encoding='utf-8'))
        if not same_fingerprint(fp, cached_fp):
            print('graph cache fingerprint changed; rebuilding relation graph')
        else:
            print('loading cached relation graph')
            edges = []
            with edges_path.open('r', encoding='utf-8') as f:
                for line in f:
                    row = json.loads(line)
                    edges.append(GraphEdge(**row))
            return edges, build_adjacency(edges)

    if not refresh and edges_path.exists() and not fp_path.exists():
        print('graph cache has no fingerprint; rebuilding relation graph')

    raw_edges = top_similarity_edges(vectors)
    edges = normalize_edges(raw_edges)
    with edges_path.open('w', encoding='utf-8') as f:
        for edge in edges:
            row = {
                'source': edge.source,
                'target': edge.target,
                'similarity': edge.similarity,
                'normalized_similarity': edge.normalized_similarity,
                'cost': edge.cost,
            }
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    fp_path.write_text(json.dumps(fp, ensure_ascii=False, indent=2), encoding='utf-8')
    return edges, build_adjacency(edges)


edges, adjacency = build_or_load_graph(embeddings)
print('nodes:', len(docs), 'edges:', len(edges))
pd.DataFrame([
    {
        'source': docs[e.source]['path'],
        'target': docs[e.target]['path'],
        'similarity': e.similarity,
        'normalized_similarity': e.normalized_similarity,
        'cost': e.cost,
    }
    for e in edges[:10]
])

similarity blocks:   0%|          | 0/18 [00:00<?, ?it/s]

similarity blocks: 100%|██████████| 18/18 [00:01<00:00, 17.53it/s]


nodes: 9048 edges: 48634


,source,target,similarity,normalized_similarity,cost
0,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/VisualSequencer/VSCommands/VSCo...,0.775005,0.617480,0.382620
1,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/A429/windows/ig/A429IG.md,0.765091,0.600568,0.399532
2,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/A429/A429Features.md,0.790031,0.643113,0.356987
3,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/A429/windows/WindowsA429Overvie...,0.794199,0.650224,0.349876
4,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/Windows/Trace/Columns/TraceColu...,0.758386,0.589130,0.410970
5,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/A429/A429Concept.md,0.889069,0.812060,0.188040
6,CANoeCANalyzer/A429/A429.md,SampConf/A429/SampConfsA429.md,0.774339,0.616345,0.383755
7,CANoeCANalyzer/A429/A429.md,CANoeCANalyzer/A429/basicsA429/A429dbSupport.md,0.757408,0.587462,0.412638
8,CANoeCANalyzer/A429/A429Concept.md,SampConf/A429/SampConfsA429.md,0.762417,0.596007,0.404093
9,CANoeCANalyzer/A429/A429Concept.md,CANoeCANalyzer/A429/basicsA429/A429basics.md,0.734094,0.547692,0.452408


## Query embedding 与 A* 风格图召回

这里不是寻找单一终点，而是做 multi-source / multi-goal 的 A* 风格 top-k 搜索：

- 起点：query 向量直接相似度最高的 `seed_k` 个文件。
- `g_cost`：从 query seed 进入图后累计的路径代价。
- `h_cost`：当前节点与 query 的向量距离估计，即 `1 - normalized_query_similarity`。
- priority：`g_cost + alpha * h_cost`。
- 最终排序：优先图路径成本低，同时保留 direct similarity 作为解释字段。

In [7]:
def embed_query(query: str) -> np.ndarray:
    if EMBEDDING_BACKEND == 'openai':
        return embed_openai_compatible([query])[0]
    if EMBEDDING_BACKEND == 'local':
        return embed_local_sentence_transformers([query])[0]
    if EMBEDDING_BACKEND == 'tfidf':
        if tfidf_vectorizer is None:
            raise RuntimeError('TF-IDF vectorizer not loaded')
        from sklearn.preprocessing import normalize
        vec = tfidf_vectorizer.transform([query])
        vec = normalize(vec, norm='l2', copy=False)
        return vec.astype(np.float32).toarray()[0]
    raise ValueError(f'Unknown EMBEDDING_BACKEND: {EMBEDDING_BACKEND}')


def embed_queries(queries: list[str]) -> np.ndarray:
    if EMBEDDING_BACKEND == 'openai':
        return embed_openai_compatible(queries)
    if EMBEDDING_BACKEND == 'local':
        return embed_local_sentence_transformers(queries)
    if EMBEDDING_BACKEND == 'tfidf':
        if tfidf_vectorizer is None:
            raise RuntimeError('TF-IDF vectorizer not loaded')
        from sklearn.preprocessing import normalize
        vecs = tfidf_vectorizer.transform(queries)
        vecs = normalize(vecs, norm='l2', copy=False)
        return vecs.astype(np.float32).toarray()
    raise ValueError(f'Unknown EMBEDDING_BACKEND: {EMBEDDING_BACKEND}')


def normalize_query_scores(scores: np.ndarray) -> np.ndarray:
    score_min = float(scores.min())
    score_max = float(scores.max())
    denom = max(score_max - score_min, 1e-9)
    return (scores - score_min) / denom


def reconstruct_path(parent: dict[int, int | None], node: int) -> list[int]:
    path = [node]
    while parent.get(node) is not None:
        node = parent[node]  # type: ignore[assignment]
        path.append(node)
    return list(reversed(path))


def graph_astar_recall_from_vector(
    query: str,
    qvec: np.ndarray,
    top_k: int = 5,
    seed_k: int = 12,
    max_expansions: int = 800,
    alpha: float = 0.35,
) -> list[dict[str, Any]]:
    direct_scores = embeddings @ qvec
    qnorm = normalize_query_scores(direct_scores)
    seeds = np.argsort(-direct_scores)[:min(seed_k, len(docs))]

    heap: list[tuple[float, float, int]] = []
    best_g: dict[int, float] = {}
    parent: dict[int, int | None] = {}

    for seed in seeds:
        seed = int(seed)
        g = 1.0 - float(qnorm[seed])
        h = 1.0 - float(qnorm[seed])
        heapq.heappush(heap, (g + alpha * h, g, seed))
        best_g[seed] = g
        parent[seed] = None

    expansions = 0
    while heap and expansions < max_expansions:
        _, g_cost, node = heapq.heappop(heap)
        if g_cost > best_g.get(node, math.inf):
            continue
        expansions += 1

        for neighbor, edge_cost, _edge_sim in adjacency.get(node, []):
            new_g = g_cost + edge_cost
            if new_g >= best_g.get(neighbor, math.inf):
                continue
            best_g[neighbor] = new_g
            parent[neighbor] = node
            h = 1.0 - float(qnorm[neighbor])
            priority = new_g + alpha * h
            heapq.heappush(heap, (priority, new_g, neighbor))

    rows = []
    for node, g_cost in best_g.items():
        path_ids = reconstruct_path(parent, node)
        rows.append({
            'path': docs[node]['path'],
            'name': docs[node]['name'],
            'direct_similarity': float(direct_scores[node]),
            'query_similarity_norm': float(qnorm[node]),
            'graph_g_cost': float(g_cost),
            'graph_score': float(1.0 / (1.0 + g_cost)),
            'hops': len(path_ids) - 1,
            'route': [docs[idx]['path'] for idx in path_ids],
            'preview': docs[node]['preview'],
        })

    rows.sort(key=lambda r: (r['graph_g_cost'], -r['direct_similarity'], r['hops']))
    return rows[:top_k]


def graph_astar_recall(
    query: str,
    top_k: int = 5,
    seed_k: int = 12,
    max_expansions: int = 800,
    alpha: float = 0.35,
) -> list[dict[str, Any]]:
    qvec = embed_query(query)
    return graph_astar_recall_from_vector(query, qvec, top_k, seed_k, max_expansions, alpha)


RUN_EXAMPLE_QUERY = False

if RUN_EXAMPLE_QUERY:
    query = 'How to configure CANoe test report XML extended information?'
    results = graph_astar_recall(query, top_k=5)
    display(pd.DataFrame(results)[['path', 'direct_similarity', 'graph_score', 'graph_g_cost', 'hops']])
else:
    results = []
    print('example query skipped; set RUN_EXAMPLE_QUERY = True to run it')

example query skipped; set RUN_EXAMPLE_QUERY = True to run it


## 查看路径解释

下面输出每个结果是从哪个 seed 经由哪些关系边走到的。`hops=0` 表示它本身就是 query 的直接向量入口。

In [8]:
for idx, item in enumerate(results, start=1):
    print(f'[{idx}] {item["path"]}')
    print(f'    direct_similarity={item["direct_similarity"]:.4f} graph_score={item["graph_score"]:.4f} hops={item["hops"]}')
    for step in item['route']:
        print(f'      -> {step}')
    print()

## evaluate_recall.jsonl Top-K 测试

使用 `nyonic-openviking/evaluate_recall.jsonl` 中的 query 字段做召回测试。默认 `top_k=5`，并评估 simple/complex 的中英文 query。可用环境变量 `EVAL_MAX_CASES=50` 做限量试跑；不设置则跑完整 691 条记录。

In [13]:
EVAL_FILE = Path('evaluate_recall.jsonl')
EVAL_TOP_K = 15
EVAL_QUERY_FIELDS = ['simple_task_zh', 'complex_task_zh', 'simple_task_en', 'complex_task_en']
EVAL_MAX_CASES_ENV = os.getenv('EVAL_MAX_CASES')
EVAL_MAX_CASES = int(EVAL_MAX_CASES_ENV) if EVAL_MAX_CASES_ENV else None


def normalize_eval_doc_path(path: str) -> str:
    path = path.replace('\\', '/').strip()
    prefix = 'FULL_DOC_MARKDOWN/'
    if path.startswith(prefix):
        path = path[len(prefix):]
    return path


def load_eval_cases(path: Path, max_cases: int | None = None) -> list[dict[str, Any]]:
    cases = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            row['target_path'] = normalize_eval_doc_path(row['doc_path'])
            cases.append(row)
            if max_cases is not None and len(cases) >= max_cases:
                break
    return cases


def evaluate_recall_cases(cases: list[dict[str, Any]], query_fields: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    queries = []
    records = []
    available_paths = {doc['path'] for doc in docs}
    for case_idx, case in enumerate(cases):
        for field in query_fields:
            query_text = case.get(field, '')
            if not query_text:
                continue
            queries.append(query_text)
            records.append({
                'case_idx': case_idx,
                'query_field': field,
                'query': query_text,
                'target_path': case['target_path'],
                'target_in_index': case['target_path'] in available_paths,
                'category': case.get('category', ''),
            })

    print('eval cases:', len(cases), '| eval queries:', len(queries), '| top_k:', EVAL_TOP_K)
    query_vectors = embed_queries(queries)

    rows = []
    for record, qvec in tqdm(list(zip(records, query_vectors)), desc='evaluate top-5 recall'):
        results = graph_astar_recall_from_vector(record['query'], qvec, top_k=EVAL_TOP_K)
        result_paths = [item['path'] for item in results]
        hit = record['target_path'] in result_paths
        rank = result_paths.index(record['target_path']) + 1 if hit else None
        rows.append({
            **record,
            'hit_top5': hit,
            'rank': rank,
            'result_paths': result_paths,
        })

    detail_df = pd.DataFrame(rows)
    summary_df = (
        detail_df.groupby('query_field')
        .agg(
            queries=('query', 'count'),
            target_coverage=('target_in_index', 'mean'),
            recall_at_TOP_K=('hit_top5', 'mean'),
            mean_rank=('rank', 'mean'),
        )
        .reset_index()
    )
    overall = pd.DataFrame([{
        'query_field': '__overall__',
        'queries': len(detail_df),
        'target_coverage': detail_df['target_in_index'].mean(),
        'recall_at_TOP_K': detail_df['hit_top5'].mean(),
        'mean_rank': detail_df['rank'].mean(),
    }])
    summary_df = pd.concat([summary_df, overall], ignore_index=True)
    return detail_df, summary_df


RUN_EVAL_RECALL = True

if RUN_EVAL_RECALL:
    eval_cases = load_eval_cases(EVAL_FILE, EVAL_MAX_CASES)
    eval_detail_df, eval_summary_df = evaluate_recall_cases(eval_cases, EVAL_QUERY_FIELDS)
    eval_detail_path = CACHE_DIR / f'evaluate_recall_top{EVAL_TOP_K}_detail.csv'
    eval_summary_path = CACHE_DIR / f'evaluate_recall_top{EVAL_TOP_K}_summary.csv'
    eval_detail_df.to_csv(eval_detail_path, index=False, encoding='utf-8-sig')
    eval_summary_df.to_csv(eval_summary_path, index=False, encoding='utf-8-sig')
    print('saved:', eval_detail_path)
    print('saved:', eval_summary_path)
    display(eval_summary_df)
else:
    print(f'evaluate_recall skipped; set RUN_EVAL_RECALL = True to run {EVAL_TOP_K} evaluation')

eval cases: 691 | eval queries: 2764 | top_k: 15


evaluate top-5 recall: 100%|██████████| 2764/2764 [00:42<00:00, 65.66it/s]


saved: graph_recall_cache\evaluate_recall_top15_detail.csv
saved: graph_recall_cache\evaluate_recall_top15_summary.csv


,query_field,queries,target_coverage,recall_at_TOP_K,mean_rank
0,complex_task_en,691,1.0,0.973951,1.597325
1,complex_task_zh,691,1.0,0.952243,1.820669
2,simple_task_en,691,1.0,0.962373,1.587970
3,simple_task_zh,691,1.0,0.956585,1.883510
4,__overall__,2764,1.0,0.961288,1.721490


## 可抽成 agent skill 的函数入口

建议把当前 notebook 拆成一个 `full_doc_graph_recall` skill，核心职责不是直接回答问题，而是稳定返回“候选文件 + 关联路径 + 召回解释”。

推荐函数入口：

- `build_full_doc_vector_graph(root_dir, cache_dir, refresh=False)`：扫描 Markdown，构造文件级 embedding 文本，生成/加载 `document_embeddings.npy`、`documents.jsonl`、`document_vectors.faiss`、`relation_edges.jsonl`。
- `query_full_doc_graph(query, top_k=5, seed_k=12, max_expansions=800)`：对 query 做 embedding，用 FAISS 找 seed，并在文件关系图上扩展，返回 top-5 文件。
- `query_full_doc_vector(query, top_k=5)`：只走 FAISS 向量库，作为图召回的 baseline 或 fallback。
- `evaluate_recall(eval_file, top_k=5, query_fields=None, max_cases=None)`：读取 `evaluate_recall.jsonl`，输出 detail/summary CSV。
- `analyze_recall_detail(detail_csv, limit=100)`：分析未命中或低排名样本，定位是同目录竞争、总览页竞争、query 过泛还是语义漂移。

工具输出建议保持 JSON serializable，至少包含：`path`、`direct_similarity`、`graph_score`、`graph_g_cost`、`hops`、`route`、`preview`。agent 拿到候选文件后，再读取原文件内容回答，不要只凭 preview 编造。

In [ ]:
AGENT_SKILL_PROMPT = """
你是 FULL_DOC_MARKDOWN 文件关系图召回 skill。你的职责是为上层 agent 找到最可能需要阅读的文档文件，而不是直接编造最终答案。

调用策略：
1. 如果缓存不存在或用户要求 refresh，调用 build_full_doc_vector_graph(root_dir='nyonic-openviking/FULL_DOC_MARKDOWN', cache_dir='nyonic-openviking/graph_recall_cache', refresh=False)。
2. 默认使用本地 sentence-transformers Qwen3-Embedding-0.6B 后端；低显存时使用 batch_size=1、较短 max_seq_length，并依赖缓存避免重复 embedding。
3. 对用户 query 调用 query_full_doc_graph(query, top_k=5, seed_k=12, max_expansions=800)。
4. 如果图召回异常或需要 baseline 对比，调用 query_full_doc_vector(query, top_k=5)。
5. 优先返回 graph_score 高、graph_g_cost 低、direct_similarity 高的候选；但如果 route 显示同目录/同主题竞争，需要提醒上层 agent 读取多个候选文件。
6. 解释结果时必须说明 route：hops=0 表示直接向量命中；hops>0 表示从哪个 seed 沿文件关系图扩展得到。
7. 需要回答具体 API/函数/配置细节时，必须读取返回的 path 原文后再回答。

输入参数：
- query: 用户自然语言问题。
- top_k: 返回文件数量，默认 5。
- seed_k: query 直接向量入口数量，默认 12。
- max_expansions: A* 图扩展节点上限，默认 800。
- refresh: 是否重建缓存，默认 false。

输出 JSON：
[
  {
    "path": "相对 FULL_DOC_MARKDOWN 的文件路径",
    "direct_similarity": 0.0,
    "graph_score": 0.0,
    "graph_g_cost": 0.0,
    "hops": 0,
    "route": ["路径1", "路径2"],
    "preview": "文件片段"
  }
]
""".strip()

print(AGENT_SKILL_PROMPT)

## 下一步实验方向

- 把文件级节点扩展为 `文件 -> 标题章节 -> chunk` 的多层图。
- 加入显式结构边：同目录文件、父子目录、同名 `.overview.md` / `.abstract.md` / 正文文件。
- cost 可以从单一 similarity 扩展为 `semantic_cost + path_cost + type_cost`。
- 对召回结果加 rerank 模型或 LLM judge，评估 `evaluate_recall.jsonl` 上的召回率变化。

## 召回关联性结论与低召回样本分析

从当前 `evaluate_recall_top5_summary.csv` 看，整体 `recall_at_5=0.907`、`mean_rank=1.43`，说明这套文件级 embedding + 图扩展不仅能做 query 召回，也能作为文件之间的语义关联信息。更重要的是，未命中样本里大量候选仍落在同一顶层模块或相邻目录，说明图关系多数时候是在“相关文件簇”内竞争，而不是完全漂移。

基于前 100 条未命中样本的快速归因：

- `same_top_module_candidates`: 53 条，top-5 至少有同顶层模块文件，常见于总览页、导航页和主题相邻页竞争。
- `same_parent_candidates`: 27 条，top-5 里有同目录候选，说明关系信息接近，但目标文件排序没进 top-5。
- `lexically_related_other_branch`: 7 条，候选和目标有词面重叠，但落到其他分支。
- `semantic_drift_or_sparse_signal`: 10 条，query 描述和目标文件文本信号不够直接，或目标是过于宽泛的 glossary/start page。
- `short_or_under_specified_query`: 3 条，query 太短或约束不足。

主要低召回风险集中在三类：

- 模块总览/导航类文档：目标常是 `CAPLfunctions.md`、`CANoeCANalyzer.md` 这类入口页，但 query 中的具体词会把召回拉到更细的功能页。
- 同主题 sibling 竞争：错误码、IL、函数列表、工具页等主题下，多个文件语义非常近，目标页容易被同目录或同模块文件挤出 top-5。
- 泛化概念问题：例如 glossary/主从节点/仿真工具总览，query 的语义更像概念解释，向量会偏向具体概念页而不是目标入口页。

改进优先级建议：先加结构边和类型权重，而不是急着换更大的 embedding 模型。尤其应给 `.overview.md`、`.abstract.md`、start page、同目录 parent/child、文件名完全匹配或模块入口页加轻量 boost。

In [10]:
DETAIL_CSV = Path('graph_recall_cache/evaluate_recall_top5_detail.csv')
LOW_RECALL_LIMIT = 100

import ast
from pathlib import PurePosixPath


def _as_paths(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return ast.literal_eval(value)


def _top_dir(path: str) -> str:
    return path.split('/')[0] if '/' in path else path


def _parent_dir(path: str) -> str:
    return str(PurePosixPath(path).parent)


def _path_tokens(path: str) -> set[str]:
    import re
    return {token for token in re.split(r'[^a-z0-9]+', path.lower()) if len(token) > 1}


def classify_low_recall(row) -> str:
    target = row['target_path']
    results = row['result_paths_list']
    if any(_parent_dir(path) == _parent_dir(target) for path in results):
        return 'same_parent_candidates'
    if any(_top_dir(path) == _top_dir(target) for path in results):
        return 'same_top_module_candidates'
    target_tokens = _path_tokens(target)
    if any(len(target_tokens & _path_tokens(path)) >= 2 for path in results):
        return 'lexically_related_other_branch'
    if len(str(row['query'])) < 45:
        return 'short_or_under_specified_query'
    return 'semantic_drift_or_sparse_signal'


def analyze_low_recall_detail(detail_csv: Path = DETAIL_CSV, limit: int = LOW_RECALL_LIMIT):
    detail_df = pd.read_csv(detail_csv)
    detail_df['result_paths_list'] = detail_df['result_paths'].apply(_as_paths)

    misses = detail_df[detail_df['hit_top5'] == False].copy()
    low_rank_hits = detail_df[(detail_df['hit_top5'] == True) & (detail_df['rank'] >= 4)].copy()
    problem_df = pd.concat([
        misses.assign(problem_type='miss'),
        low_rank_hits.assign(problem_type='low_rank_hit'),
    ], ignore_index=True)

    # 优先分析真正未命中；未命中之后再看 rank 4/5 的低排名命中。
    problem_df['problem_priority'] = problem_df['problem_type'].map({'miss': 0, 'low_rank_hit': 1})
    problem_df['rank_sort'] = problem_df['rank'].fillna(99)
    problem_df = problem_df.sort_values(['problem_priority', 'rank_sort'], ascending=[True, False]).head(limit).copy()

    problem_df['reason_bucket'] = problem_df.apply(classify_low_recall, axis=1)
    problem_df['target_top_dir'] = problem_df['target_path'].apply(_top_dir)
    problem_df['target_parent'] = problem_df['target_path'].apply(_parent_dir)
    problem_df['top1_path'] = problem_df['result_paths_list'].apply(lambda paths: paths[0] if paths else '')
    problem_df['any_same_parent'] = problem_df.apply(
        lambda row: any(_parent_dir(path) == row['target_parent'] for path in row['result_paths_list']),
        axis=1,
    )
    problem_df['any_same_top_dir'] = problem_df.apply(
        lambda row: any(_top_dir(path) == row['target_top_dir'] for path in row['result_paths_list']),
        axis=1,
    )

    summary = {
        'sampled_problem_rows': len(problem_df),
        'total_misses': int((detail_df['hit_top5'] == False).sum()),
        'total_low_rank_hits_rank_4_or_5': int(((detail_df['hit_top5'] == True) & (detail_df['rank'] >= 4)).sum()),
        'reason_bucket_counts': problem_df['reason_bucket'].value_counts().to_dict(),
        'query_field_counts': problem_df['query_field'].value_counts().to_dict(),
        'category_counts_top10': problem_df['category'].value_counts().head(10).to_dict(),
        'target_top_dir_counts_top10': problem_df['target_top_dir'].value_counts().head(10).to_dict(),
        'same_parent_ratio': float(problem_df['any_same_parent'].mean()),
        'same_top_dir_ratio': float(problem_df['any_same_top_dir'].mean()),
    }

    output_csv = CACHE_DIR / 'evaluate_recall_low100_analysis.csv'
    problem_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    return summary, problem_df, output_csv


low_recall_summary, low_recall_100_df, low_recall_output_csv = analyze_low_recall_detail()
print(json.dumps(low_recall_summary, ensure_ascii=False, indent=2))
print('saved:', low_recall_output_csv)
low_recall_100_df[[
    'problem_type', 'query_field', 'category', 'reason_bucket',
    'target_path', 'top1_path', 'result_paths'
]].head(20)

{
  "sampled_problem_rows": 100,
  "total_misses": 256,
  "total_low_rank_hits_rank_4_or_5": 113,
  "reason_bucket_counts": {
    "same_top_module_candidates": 53,
    "same_parent_candidates": 27,
    "semantic_drift_or_sparse_signal": 10,
    "lexically_related_other_branch": 7,
    "short_or_under_specified_query": 3
  },
  "query_field_counts": {
    "simple_task_zh": 34,
    "simple_task_en": 32,
    "complex_task_zh": 19,
    "complex_task_en": 15
  },
  "category_counts_top10": {
    "查询模块总览与导航": 50,
    "查询核心概念与技术细节": 14,
    "查询具体的函数/方法": 14,
    "查询针对特定复杂任务提供的实现方案说明(Guidelines / Examples)": 6,
    "查询类的构造和定义": 5,
    "查询模块总览 with 导航": 4,
    "查询具体的工具/功能": 3,
    "查询交互层/IL的使用": 2,
    "查询事件规程/事件处理程序": 1,
    "查询选择器": 1
  },
  "target_top_dir_counts_top10": {
    "CAPLFunctions": 60,
    "CANoeCANalyzer": 27,
    "SampConf": 6,
    "TestCommands": 4,
    "Shared": 3
  },
  "same_parent_ratio": 0.27,
  "same_top_dir_ratio": 0.8
}
saved: graph_recall_cache\evaluate_recall_low100_

,problem_type,query_field,category,reason_bucket,target_path,top1_path,result_paths
0,miss,simple_task_zh,查询核心概念与技术细节,same_top_module_candidates,CAPLFunctions/Glossary.md,CANoeCANalyzer/General/HELP_NETNODES.md,"['CANoeCANalyzer/General/HELP_NETNODES.md', 'C..."
1,miss,complex_task_zh,查询核心概念与技术细节,semantic_drift_or_sparse_signal,CAPLFunctions/Glossary.md,CANoeCANalyzer/MOST/MOST_Simulation_Concept/mo...,['CANoeCANalyzer/MOST/MOST_Simulation_Concept/...
2,miss,simple_task_zh,查询模块总览与导航,same_top_module_candidates,CAPLFunctions/CAPLfunctions.md,CAPLFunctions/Other/CAPLGeneralStartPage.md,['CAPLFunctions/Other/CAPLGeneralStartPage.md'...
3,miss,simple_task_en,查询模块总览与导航,same_top_module_candidates,CAPLFunctions/CAPLfunctions.md,CAPLFunctions/Other/CAPLGeneralStartPage.md,['CAPLFunctions/Other/CAPLGeneralStartPage.md'...
4,miss,simple_task_zh,查询模块总览与导航,same_top_module_candidates,CANoeCANalyzer/CANoeCANalyzer.md,COMInterface/Properties/COMPropertySimulation.md,['COMInterface/Properties/COMPropertySimulatio...
5,miss,complex_task_zh,查询模块总览与导航,same_top_module_candidates,CANoeCANalyzer/CANoeCANalyzer.md,CANoeCANalyzer/ISO11783/SimulationAndTest.md,['CANoeCANalyzer/ISO11783/SimulationAndTest.md...
6,miss,simple_task_en,查询模块总览与导航,same_top_module_candidates,CANoeCANalyzer/CANoeCANalyzer.md,COMInterface/Properties/COMPropertySimulation.md,['COMInterface/Properties/COMPropertySimulatio...
7,miss,complex_task_en,查询模块总览与导航,same_top_module_candidates,CANoeCANalyzer/CANoeCANalyzer.md,CANoeCANalyzer/ISO11783/SimulationAndTest.md,['CANoeCANalyzer/ISO11783/SimulationAndTest.md...
8,miss,simple_task_zh,查询模块总览与导航,same_top_module_candidates,CANoeCANalyzer/CANoeCANalyzerTools.md,SampConf/ISO11783/CANoe/MoreExamples/Auxiliary...,['SampConf/ISO11783/CANoe/MoreExamples/Auxilia...
9,miss,complex_task_zh,查询模块总览与导航,same_top_module_candidates,CANoeCANalyzer/CANoeCANalyzerTools.md,CANoeCANalyzer/Ribbon/File/Options/ExternalPro...,['CANoeCANalyzer/Ribbon/File/Options/ExternalP...


## 无模型 Rerank Top-K 实验

Rerank 不一定必须额外模型。这里先做无模型 rerank：只使用 query 文本、候选路径、候选原始 rank 做轻量融合，不重新 embedding，也不加载 Qwen 或 cross-encoder。

这类 rerank 的目标是验证路径/文件名/入口页等结构信号能不能把 top-N 候选重排成更好的 top-k。注意：如果原始图召回排序已经很强，无模型 rerank 可能不会提升，甚至会因为启发式过强而降低效果。因此这个 block 会同时输出 baseline 和 rerank 指标。

In [12]:
from pathlib import Path, PurePosixPath
import ast
import json
import re
import pandas as pd

CACHE_DIR = Path('graph_recall_cache')
RERANK_INPUT_DETAIL = CACHE_DIR / 'evaluate_recall_top20_detail.csv'
RERANK_TOP_K = 15
RERANK_OUTPUT_DETAIL = CACHE_DIR / f'evaluate_recall_rerank_from_top20_top{RERANK_TOP_K}_detail.csv'
RERANK_OUTPUT_SUMMARY = CACHE_DIR / f'evaluate_recall_rerank_from_top20_top{RERANK_TOP_K}_summary.csv'

# 原始 rank 仍然是主信号；其他无模型特征只给很小 boost，避免启发式压过图召回。
RERANK_WEIGHTS = {
    'base_rank': 2.0,
    'path_overlap': 0.20,
    'stem_overlap': 0.30,
    'nav_entry': 0.06,
    'nav_penalty_function_leaf': 0.02,
    'function_leaf': 0.04,
    'function_overview_penalty': 0.02,
    'error_signal': 0.05,
    'tool_signal': 0.05,
    'shallow_path': 0.02,
}

NAV_QUERY_HINTS = (
    '总览', '概览', '分类', '导航', '有哪些', '哪些', '核心功能', '常见任务', '基础知识', '首页', '列表', '所有', '基本定义', '区别',
    'overview', 'classification', 'categorical', 'categories', 'available', 'what ', 'which ', 'core functions', 'common tasks', 'basic', 'definition', 'differences', 'start page', 'home page', 'list',
)
FUNCTION_QUERY_HINTS = ('函数', '方法', '回调', '事件', '返回值', '参数', '调用', '接口', 'function', 'method', 'callback', 'event', 'return', 'parameter', 'call', 'api')
ERROR_QUERY_HINTS = ('错误', '错误码', '异常', '超时', '未找到', 'error', 'errors', 'timeout', 'not found', 'exception', 'code', 'codes')
TOOL_QUERY_HINTS = ('工具', '编辑器', '配置管理', '启动', 'tool', 'tools', 'editor', 'configuration manager', 'launch')


def parse_result_paths(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return ast.literal_eval(value)


def text_tokens(value: str) -> set[str]:
    return {token for token in re.split(r'[^a-z0-9]+', str(value).lower()) if len(token) > 1}


def path_tokens(path: str) -> set[str]:
    return text_tokens(path)


def stem_tokens(path: str) -> set[str]:
    return text_tokens(PurePosixPath(path).stem)


def contains_any(text: str, hints: tuple[str, ...]) -> bool:
    lower = str(text).lower()
    return any(hint.lower() in lower for hint in hints)


def no_model_rerank_score(query: str, path: str, original_rank_zero_based: int, weights: dict[str, float] = RERANK_WEIGHTS) -> float:
    query_token_set = text_tokens(query)
    path_token_set = path_tokens(path)
    stem_token_set = stem_tokens(path)
    path_lower = path.lower()
    stem_lower = PurePosixPath(path).stem.lower()

    score = weights['base_rank'] / (original_rank_zero_based + 1)
    if query_token_set:
        score += weights['path_overlap'] * len(query_token_set & path_token_set) / len(query_token_set)
        score += weights['stem_overlap'] * len(query_token_set & stem_token_set) / len(query_token_set)

    is_nav_query = contains_any(query, NAV_QUERY_HINTS)
    is_function_query = contains_any(query, FUNCTION_QUERY_HINTS)
    is_error_query = contains_any(query, ERROR_QUERY_HINTS)
    is_tool_query = contains_any(query, TOOL_QUERY_HINTS)

    if is_nav_query:
        if any(marker in stem_lower for marker in ('overview', 'startpage', 'start', 'glossary')):
            score += weights['nav_entry']
        if stem_lower.endswith('functions') or stem_lower.endswith('canoecanoalyzer'):
            score += weights['nav_entry'] * 0.8
        if '/functions/' in path_lower:
            score -= weights['nav_penalty_function_leaf']

    if is_function_query:
        if '/functions/' in path_lower or stem_lower.startswith('caplfunction'):
            score += weights['function_leaf']
        if any(marker in stem_lower for marker in ('overview', 'startpage')):
            score -= weights['function_overview_penalty']

    if is_error_query and any(marker in path_lower for marker in ('error', 'errorcodes', 'lasterror')):
        score += weights['error_signal']

    if is_tool_query and any(marker in path_lower for marker in ('tool', 'editor', 'browser')):
        score += weights['tool_signal']

    score += weights['shallow_path'] / (path.count('/') + 1)
    return float(score)


def rerank_paths_no_model(query: str, paths: list[str]) -> list[str]:
    scored = [
        (no_model_rerank_score(query, path, idx), idx, path)
        for idx, path in enumerate(paths)
    ]
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [path for _, _, path in scored]


def evaluate_no_model_rerank(detail_csv: Path = RERANK_INPUT_DETAIL, top_k: int = RERANK_TOP_K):
    detail_df = pd.read_csv(detail_csv)
    detail_df['candidate_paths'] = detail_df['result_paths'].apply(parse_result_paths)

    rows = []
    for _, row in detail_df.iterrows():
        original_paths = row['candidate_paths']
        reranked_paths = rerank_paths_no_model(row['query'], original_paths)

        baseline_hit = row['target_path'] in original_paths[:top_k]
        baseline_rank = original_paths.index(row['target_path']) + 1 if row['target_path'] in original_paths else None
        rerank_hit = row['target_path'] in reranked_paths[:top_k]
        rerank_rank = reranked_paths.index(row['target_path']) + 1 if row['target_path'] in reranked_paths else None

        rows.append({
            'case_idx': row['case_idx'],
            'query_field': row['query_field'],
            'query': row['query'],
            'target_path': row['target_path'],
            'category': row.get('category', ''),
            'baseline_hit_topk': baseline_hit,
            'baseline_rank': baseline_rank,
            'rerank_hit_topk': rerank_hit,
            'rerank_rank': rerank_rank,
            'baseline_paths_topk': original_paths[:top_k],
            'reranked_paths_topk': reranked_paths[:top_k],
            'candidate_count': len(original_paths),
        })

    rerank_df = pd.DataFrame(rows)
    summary_df = (
        rerank_df.groupby('query_field')
        .agg(
            queries=('query', 'count'),
            baseline_recall_at_k=('baseline_hit_topk', 'mean'),
            rerank_recall_at_k=('rerank_hit_topk', 'mean'),
            baseline_mean_rank=('baseline_rank', 'mean'),
            rerank_mean_rank=('rerank_rank', 'mean'),
        )
        .reset_index()
    )
    overall = pd.DataFrame([{
        'query_field': '__overall__',
        'queries': len(rerank_df),
        'baseline_recall_at_k': rerank_df['baseline_hit_topk'].mean(),
        'rerank_recall_at_k': rerank_df['rerank_hit_topk'].mean(),
        'baseline_mean_rank': rerank_df['baseline_rank'].mean(),
        'rerank_mean_rank': rerank_df['rerank_rank'].mean(),
    }])
    summary_df = pd.concat([summary_df, overall], ignore_index=True)

    rerank_df.to_csv(RERANK_OUTPUT_DETAIL, index=False, encoding='utf-8-sig')
    summary_df.to_csv(RERANK_OUTPUT_SUMMARY, index=False, encoding='utf-8-sig')
    return rerank_df, summary_df


rerank_detail_df, rerank_summary_df = evaluate_no_model_rerank()
print('saved:', RERANK_OUTPUT_DETAIL)
print('saved:', RERANK_OUTPUT_SUMMARY)
rerank_summary_df

saved: graph_recall_cache\evaluate_recall_rerank_from_top20_top15_detail.csv
saved: graph_recall_cache\evaluate_recall_rerank_from_top20_top15_summary.csv


,query_field,queries,baseline_recall_at_k,rerank_recall_at_k,baseline_mean_rank,rerank_mean_rank
0,complex_task_en,691,0.959479,0.960926,1.709145,1.701649
1,complex_task_zh,691,0.947902,0.947902,2.065250,2.072838
2,simple_task_en,691,0.960926,0.960926,1.676692,1.679699
3,simple_task_zh,691,0.958032,0.956585,1.972851,2.012066
4,__overall__,2764,0.956585,0.956585,1.855313,1.865863
